# Web Scraping

**Group members:**

- **José Jesus** - 20251389
- **Lara Barradas** - 20251397
- **Lara Manso** - 20251364
- **Zahra Ahamed** - 20251382

**Introduction**

This section of the project shows how data about the locations and coordinates of the birthplaces of different artists were obtained, for future use in the creation of Interactive Maps with such information.

All the data was obtained using Wikipedia as a source and following their Terms and Conditions in order to have a proper, legal use of the website. Furthermore, not every artist has the information required in their Wikipedia page, meaning some may miss this information.

**Importing the libraries needed and defining the settings for the Web Scraping**

In [1]:
import re
import time
import unicodedata

import pandas as pd
import requests
from bs4 import BeautifulSoup

WIKI_BASE = "https://en.wikipedia.org"
WIKI_MAIN_PAGE = WIKI_BASE + "/wiki/Main_Page"    
WIKI_SEARCH_URL = WIKI_BASE + "/w/index.php"      
PAUSE_SECONDS = 0.5    

wiki_session = requests.Session()     # reusable connection to Wikipedia
wiki_session.headers.update(      # identifying ourselves to Wikipedia so as to agree with Wikipedia's usage terms
    {"User-Agent": "NOVA-IMS-PDS-student-project (20251389@novaims.unl.pt)"}
)

MONTHS = ("January|February|March|April|May|June|July|August|"
          "September|October|November|December")
DATE_PATTERN = re.compile(rf"\b({MONTHS})\b|\bage\s*\d+|\b\d{{4}}\b", re.I)

For the settings, the base address of Wikipedia was defined in "WIKI_BASE". "WIKI_MAIN_PAGE" adds "/wiki/Main_Page" to it, which is the starting point required by the assignment and is opened in a later cell. "WIKI_SEARCH_URL" is the address used to search for artists: it starts with Wikipedia's usual search address and is replaced later by the one read from the search box of the Main Page. 

In PAUSE_SECONDS, a 0.5 buffer was designed, so the code waits half a second after every successful download, preventing errors when running the code or Wikipedia blocking our usage of the website.

In addition, "wiki_session" is the variable that keeps a reusable connection open and, in order to identify ourselves and be allowed usage of the website by Wikipedia, the email of one of the students of the project was provided in case contact is needed.

The months were defined in "MONTHS" and "DATE_PATTERN" uses "MONTHS" to detect lines that look like a date. It accepts a month name: "\b" is used as a word boundary ("May" different from "Mayfield"), only taking the month by itself. It also accepts "age" followed by numbers: "\bage\s*\d+" (eg.: age 39). "DATE_PATTERN" also recognizes exactly 4 digits as a year (e.g.: 2009).  

The libraries imported and the reason for their usage is seen below:

**re**: Wikipedia text is messy, and regular expressions let us find patterns (dates, citation marks, coordinates) in it.

**time**: Pausing between requests so Wikipedia doesn’t block us.

**unicodedata**: Names with accents must match their Wikipedia titles (“Beyoncé” vs “Beyonce”).

**pandas**: Handles the dataset; reading, merging and saving tables.

**requests**: Downloads the web pages.

**BeautifulSoup (bs4)**: Turns the downloaded HTML text into a structure we can search and lets us ask for exactly what we want.

The versions of the packages (external python libraries) are as follows:

In [2]:
import sys
from importlib.metadata import version

print("PACKAGE VERSIONS")
print("python:", sys.version.split()[0])
for package_name in ["pandas", "requests", "beautifulsoup4"]:
    print(f"{package_name}: {version(package_name)}")

PACKAGE VERSIONS
python: 3.13.15
pandas: 3.0.6
requests: 2.34.2
beautifulsoup4: 4.15.0


**Functions to download and search Wikipedia pages**

In [3]:
def fetch_wiki_soup(url, params=None, retries=3):
    # download a Wikipedia page, returns (soup, final_url) or (None, None)
    for attempt in range(retries):
        try:
            response = wiki_session.get(url, params=params, timeout=15)
            if response.status_code == 200:
                time.sleep(PAUSE_SECONDS)
                return BeautifulSoup(response.text, "html.parser"), response.url
            if response.status_code == 429:           # if there are too many requests, asks python to wait longer
                time.sleep(5 * (attempt + 1))
        except requests.RequestException:
            time.sleep(2)
    return None, None


def search_wikipedia(query):
    # same as typing in the search box of the Main Page
    return fetch_wiki_soup(
        WIKI_SEARCH_URL,                                   
        params={"search": query, "title": "Special:Search", "ns0": 1},
    )


def get_search_result_urls(search_soup, max_results=3):
    result_links = search_soup.select("li.mw-search-result .mw-search-result-heading a")
    return [WIKI_BASE + link["href"] for link in result_links[:max_results]]

- **"fetch_wiki_soup"** downloads a page and tries up to 3 times. If the download works (code 200), it waits "PAUSE_SECONDS" (0.5 seconds) and returns the page already parsed by BeautifulSoup, together with its final address. If Wikipedia answers "too many requests" (code 429), the code waits longer before trying again and network errors are caught so a single failure doesn't stop the whole run.

- **"search_wikipedia"** searches Wikipedia using the address "WIKI_SEARCH_URL", which is defined in the next cell from the search box of the Main Page, so no address of an article is written by hand.

- **"get_search_result_urls"** takes the first 3 results of a search, which are the best candidates for the artist's page.

**Starting point: the Wikipedia Main Page**

In [4]:
main_page_soup, main_page_final_url = fetch_wiki_soup(WIKI_MAIN_PAGE)     # opens Wikipedia Main Page

# finds the search box of the Main Page and reads where it sends the query
if main_page_soup is not None:
    main_page_search_form = main_page_soup.find("form", id="searchform") 
else: 
    main_page_search_form = None
if main_page_search_form is not None:
    search_form_action = main_page_search_form.get("action")
else:
    search_form_action = None

# uses that address for all the searches (otherwise keeps the default one from the first cell)
if search_form_action and search_form_action.startswith("/"):
    WIKI_SEARCH_URL = WIKI_BASE + search_form_action

print("MAIN PAGE")
print("Main Page opened:", main_page_soup is not None)
print("Address opened:", main_page_final_url)
print("Search box found on the Main Page:", search_form_action is not None)
print("Search address used:", WIKI_SEARCH_URL)

MAIN PAGE
Main Page opened: True
Address opened: https://en.wikipedia.org/wiki/Main_Page
Search box found on the Main Page: True
Search address used: https://en.wikipedia.org/w/index.php


The assignment requires the scraping to start from the Wikipedia Main Page. For this reason, the code first opens the Main Page ("WIKI_MAIN_PAGE") and looks for its search box (the form with the id "searchform"), reading the address to which it sends the search (the "action" of the form). That address is saved in "WIKI_SEARCH_URL" and is the one used by "search_wikipedia" for every search. This means that each artist is found through the search function of the Main Page, and that no address of an article is written by hand. From the article found, the birthplace or origin is extracted and, from the page of that place, its coordinates, in this order, as the assignment asks.

If the Main Page or its search box cannot be found, the default search address defined in the first cell is kept, so the scraper keeps working. The prints above show whether the Main Page was opened and whether the search box was found.

When the code was run, the Main Page was opened correctly and its search box was found on it. The address read from the search box was "https://en.wikipedia.org/w/index.php", which is the same as the default address defined in the first cell, confirming that the starting point required in the assignment was respected.

**Functions to find the correct Wikipedia page of each artist**

In [5]:
# exact Wikipedia titles for artists whose name is shared with another artist (see the country check review)
WIKIPEDIA_TITLE_OVERRIDES = {"Dave": "Dave (rapper)",
                            "SEVENTEEN": "Seventeen (South Korean band)"}

# symbols that artists use in place of letters or words (see the review of the weak matches)
SYMBOL_WORDS = {"$": "s", "&": " and ", "+": " and ", "!": "i"}

def normalize_name(text):
    # removes brackets such as "(musician)", writes symbols as letters ($ -> s, & -> and)
    text = re.sub(r"\s*\(.*?\)", "", text)
    for symbol, word in SYMBOL_WORDS.items():
        text = text.replace(symbol, word)
    # turns the text to lowercase, removes accents and keeps only letters and digits
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]", "", text.lower())


def get_match_quality(artist_name, page_names):
    # "strong": a name of the page is exactly the artist's name
    # "weak": one name contains the other (e.g. "ZAYN" and "Zayn Malik"); None: different names
    artist_key = normalize_name(artist_name)
    if not artist_key:
        return None
    page_keys = [normalize_name(page_name) for page_name in page_names]
    page_keys = [page_key for page_key in page_keys if page_key]
    if artist_key in page_keys:
        return "strong"
    for page_key in page_keys:
        if min(len(artist_key), len(page_key)) >= 4 and (artist_key in page_key or page_key in artist_key):
            return "weak"
    return None


def get_music_infobox(article_soup):
    # returns the infobox only if it belongs to a musician or a band
    for infobox in article_soup.find_all("table", class_="infobox"):
        row_headers = {th.get_text(" ", strip=True) for th in infobox.find_all("th")}
        is_music_infobox = bool(row_headers & {"Genres", "Labels", "Instruments"})
        is_release = "Released" in row_headers          # albums and songs have a "Released" row
        if is_music_infobox and not is_release:
            return infobox
    return None


def get_page_title(article_soup):
    title_tag = article_soup.find(id="firstHeading")
    if title_tag:
        return title_tag.get_text(" ", strip=True) 
    else:
        return ""


def get_lead_bold_names(article_soup):
    # names in bold in the first paragraph: Wikipedia writes the artist's real and stage names there
    for paragraph in article_soup.select("div.mw-parser-output > p"):
        if paragraph.get_text(strip=True):
            return [bold_tag.get_text(" ", strip=True) for bold_tag in paragraph.find_all("b")]
    return []


def find_artist_article(artist_name, artist_type):
    # searches Wikipedia and returns (article_url, infobox, match_quality)
    if artist_name in WIKIPEDIA_TITLE_OVERRIDES:
        search_queries = [WIKIPEDIA_TITLE_OVERRIDES[artist_name]]       # exact title, checked by hand
    else:
        if artist_type == "Group":
            kind_word = "band"
        else:
            kind_word = "singer"
        search_queries = [f"{artist_name} {kind_word}", f"{artist_name} musician", artist_name]
    weak_candidate = None

    for query in search_queries:
        search_soup, final_url = search_wikipedia(query)
        if search_soup is None:
            continue

        # if it is an exact title, Wikipedia redirects straight to the article
        if search_soup.find("ul", class_="mw-search-results") is None:
            candidates = [(final_url, search_soup)]
        else:
            candidates = []
            for candidate_url in get_search_result_urls(search_soup):
                candidate_soup, candidate_final = fetch_wiki_soup(candidate_url)
                if candidate_soup is not None:
                    candidates.append((candidate_final, candidate_soup))

        for candidate_url, candidate_soup in candidates:
            infobox = get_music_infobox(candidate_soup)
            if infobox is None:
                continue
            page_names = [get_page_title(candidate_soup)] + get_lead_bold_names(candidate_soup)
            match_quality = get_match_quality(artist_name, page_names)
            if match_quality == "strong":
                return candidate_url, infobox, "strong"
            if match_quality == "weak" and weak_candidate is None:
                weak_candidate = (candidate_url, infobox, "weak")

    return weak_candidate if weak_candidate else (None, None, None)

The main risk in scraping is matching an artist to the wrong page (e.g. a different person with the same name, or an album of the artist). These functions reduce that risk:

- **"WIKIPEDIA_TITLE_OVERRIDES"**: for the two artists whose name is shared with another artist ("Dave" and "SEVENTEEN"), the exact Wikipedia title is given to the search box. This list was created after the manual review described in the results section.

- **"normalize_name"**: removes brackets such as "(musician)", writes the symbols that some artists use in place of letters ("$" as "s", "&" and "+" as "and", "!" as "i"), converts the name to lowercase, removes accents and keeps only letters and digits. For example, "A$AP Rocky" and "ASAP Rocky" both become "asaprocky", and "Beyoncé" and "Beyonce" both become "beyonce".

- **"get_match_quality"**: compares the artist's name with the names of a page. It returns "strong" when a name of the page is exactly the artist's name, and "weak" when one name contains the other (e.g. "ZAYN" and "Zayn Malik"). Any other page is rejected. A minimum of 4 characters avoids short names matching by chance.

- **"get_music_infobox"**: a page only counts if its infobox has a row such as "Genres", "Labels" or "Instruments", which filters out people who are not musicians. Infoboxes with a "Released" row are rejected, because they belong to albums and songs.

- **"get_page_title"**: reads the title of the article.

- **"get_lead_bold_names"**: reads the names written in bold in the first paragraph of the article, where Wikipedia writes the artist's real and stage names. These names are compared together with the title.

- **"find_artist_article"**: tries three searches in order: the name with "singer" (or "band" for groups, using the column "Artist Type"), the name with "musician" and the name alone. If Wikipedia redirects straight to an article, that article is checked; otherwise, the first 3 results are checked. A strong match is returned immediately, and a weak match is only used if no strong one is found. This classification is saved in the column "Match Quality" (function "scrape_artist_location"), so the weak matches can be reviewed by hand.

**Functions to extract the birthplace or origin from the infobox**

In [6]:
def clean_text(text):
    text = re.sub(r"\[\w+\]", "", text)          # remove [1], [a]
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+,", ",", text)             # "Los Angeles , California" -> "Los Angeles, California"
    return text.strip(" ,;")


def get_place_from_cell(cell, from_start):
    # return (place_text, place_links) from an infobox 'Born' / 'Origin' cell
    birthplace_tag = cell.find(class_="birthplace")
    container = None

    if birthplace_tag is not None:
        container = BeautifulSoup(str(birthplace_tag), "html.parser")
    else:
        # split the cell on <br>. The name, the date and the place are separated by <br>
        # Origin: use the first segment that is not a date; 
        # Born: use the last segment that comes after the date line (so the name is never returned).
        segments = [[]]
        for node in cell.contents:
            if getattr(node, "name", None) == "br":
                segments.append([])
            else:
                segments[-1].append(node)
        segment_texts = [
            BeautifulSoup("".join(str(n) for n in seg), "html.parser").get_text(" ", strip=True)
            for seg in segments
        ]
        if from_start:
            ordered_segments = segments                      # Origin: first non-date segment
        else:
            # Born: [name] <br> [date] <br> [place] -> the place must come after a date
            date_positions = [i for i, t in enumerate(segment_texts) if DATE_PATTERN.search(t)]
            last_date_position = date_positions[-1] if date_positions else -1
            ordered_segments = segments[last_date_position + 1:][::-1] if date_positions else []
        for segment in ordered_segments:
            segment_soup = BeautifulSoup("".join(str(n) for n in segment), "html.parser")
            for citation in segment_soup.find_all("sup"):
                citation.decompose()
            segment_text = clean_text(segment_soup.get_text(" ", strip=True))
            if segment_text and not DATE_PATTERN.search(segment_text):
                container = segment_soup
                break

    if container is None:
        return None, []

    for citation in container.find_all("sup"):
        citation.decompose()
    place_text = clean_text(container.get_text(" ", strip=True))
    if not place_text or DATE_PATTERN.search(place_text):
        return None, []

    # Links in order: city, state, country (used later as fallbacks for coordinates)
    place_links = []
    for link in container.find_all("a", href=True):
        href = link["href"]
        if href.startswith("/wiki/") and ":" not in href and WIKI_BASE + href not in place_links:
            place_links.append(WIKI_BASE + href)
    return place_text, place_links


def extract_birthplace(infobox, artist_type):
    # groups -> 'Origin' first; solo artists -> 'Born' first
    if artist_type == "Group":
        preferred_labels = ["Origin", "Born"] 
    else: 
        preferred_labels = ["Born", "Origin"]
    infobox_rows = {}
    for row in infobox.find_all("tr"):
        header, value = row.find("th"), row.find("td")
        if header and value:
            infobox_rows.setdefault(header.get_text(" ", strip=True), value)

    for label in preferred_labels:
        if label in infobox_rows:
            place_text, place_links = get_place_from_cell(
                infobox_rows[label], from_start=(label == "Origin")
            )
            if place_text:
                return label, place_text, place_links
    return None, None, []

In the infobox, the "Born" cell mixes the name, the birth date and the birthplace on separate lines, while groups usually have an "Origin" row with only a place. These functions isolate the place:

- **"clean_text"**: removes citation marks such as [1], extra spaces and stray commas ("Los Angeles , California" becomes "Los Angeles, California").

- **"get_place_from_cell"**: first looks for the tag that Wikipedia uses for birthplaces ("birthplace"). If it doesn't exist, it splits the cell into lines and uses "DATE_PATTERN" to discard dates. In a "Born" cell the place must come after the date line, which avoids returning the person's name when no place is written. In an "Origin" cell the first line without a date is used. It also collects the links inside the place (city, state, country), which are later used to find coordinates.

- **"extract_birthplace"**: looks for "Born" first for solo artists and "Origin" first for groups, using the other row as a backup.

**Functions to obtain the coordinates of each birthplace**

In [7]:
def dms_to_decimal(dms_text):
    # 43°42′N -> 43.7 (south and west become negative)
    match = re.search(r"(\d+(?:\.\d+)?)°(?:(\d+)[′'])?(?:(\d+(?:\.\d+)?)[″\"])?\s*([NSEW])", dms_text)
    if not match:
        return None
    degrees, minutes, seconds, hemisphere = match.groups()
    value = float(degrees) + int(minutes or 0) / 60 + float(seconds or 0) / 3600
    return -value if hemisphere in "SW" else value


def extract_coordinates(place_soup):
    # read the coordinates of a place page (title coordinates, else infobox)
    search_areas = [place_soup.find(id="coordinates"), place_soup.find("table", class_="infobox")]
    for area in search_areas:
        if area is None:
            continue
        geo_tag = area.find("span", class_="geo")          # decimal: "43.7001; -79.4163" (directly from Wikipedia HTML)
        if geo_tag:
            try:
                latitude_text, longitude_text = geo_tag.get_text().split(";")
                return float(latitude_text), float(longitude_text)
            except ValueError:
                pass
        latitude_tag = area.find("span", class_="latitude")  # degrees/minutes version
        longitude_tag = area.find("span", class_="longitude")
        if latitude_tag and longitude_tag:
            latitude = dms_to_decimal(latitude_tag.get_text())  # decimal (through 'dms_to_decimal' function)
            longitude = dms_to_decimal(longitude_tag.get_text())
            if latitude is not None and longitude is not None:
                return latitude, longitude
    return None, None


def get_place_coordinates(place_links, place_text, coordinate_cache):
    # try the city link first, then broader links (state, country) and then finishing with a search
    for place_url in place_links:
        if place_url not in coordinate_cache:              # cache: each place is read once
            place_soup, _ = fetch_wiki_soup(place_url)
            coordinate_cache[place_url] = (
                extract_coordinates(place_soup) if place_soup is not None else (None, None)
            )
        latitude, longitude = coordinate_cache[place_url]
        if latitude is not None:
            return latitude, longitude, place_url

    # fallback: search the place text on Wikipedia
    search_soup, final_url = search_wikipedia(place_text)
    if search_soup is not None:
        if search_soup.find("ul", class_="mw-search-results") is None:
            latitude, longitude = extract_coordinates(search_soup)
            if latitude is not None:
                return latitude, longitude, final_url
        else:
            for result_url in get_search_result_urls(search_soup, max_results=1):
                result_soup, result_final = fetch_wiki_soup(result_url)
                if result_soup is not None:
                    latitude, longitude = extract_coordinates(result_soup)
                    if latitude is not None:
                        return latitude, longitude, result_final
    return None, None, None

- **"dms_to_decimal"**: converts coordinates written in degrees and minutes (43°42′N) into decimal numbers (43.7). South and West become negative, which is the format needed to draw maps.

- **"extract_coordinates"**: reads the coordinates of a place page. It looks first at the coordinates in the page title and then in the infobox, so unrelated coordinates elsewhere on the page are not used. Wikipedia shows coordinates in degrees and minutes, but also keeps a hidden decimal copy in the page's HTML, which is easier to read, so the function tries that first. If the decimal copy is missing, it reads the visible degrees and minutes and converts them with "dms_to_decimal".

- **"get_place_coordinates"**: tries the links of the place in order (city, state, country, respectively), so an artist still gets coordinates when the city page has none and the page that supplied them is recorded. As a last option, it searches Wikipedia for the place text. While all of this is happening, a dictionary ("coordinate_cache") stores places already visited, making it so that a city such as Los Angeles is downloaded only once, which saves time and reduces requests to Wikipedia.

**Functions that combine all the steps for one artist/band and for the whole dataset**

In [8]:
def scrape_artist_location(artist_name, artist_type, coordinate_cache):
    # birthplace/origin + coordinates of one artist/band
    result = {
        "Birthplace": None, "Latitude": None, "Longitude": None,
        "Wikipedia Page": None, "Coordinates From": None,
        "Match Quality": None, "Scrape Status": "no_article",
    }
    article_url, infobox, match_quality = find_artist_article(artist_name, artist_type)
    if infobox is None:
        return result
    result.update({"Wikipedia Page": article_url, "Match Quality": match_quality,
                   "Scrape Status": "no_birthplace"})

    _, place_text, place_links = extract_birthplace(infobox, artist_type)
    if place_text is None:
        return result
    result.update({"Birthplace": place_text, "Scrape Status": "no_coordinates"})

    latitude, longitude, coordinates_url = get_place_coordinates(
        place_links, place_text, coordinate_cache
    )
    if latitude is not None:
        result.update({"Latitude": latitude, "Longitude": longitude,
                       "Coordinates From": coordinates_url, "Scrape Status": "ok"})
    return result


def add_locations_to_dataset(artists_dataset, progress_path="scrape_progress.csv", save_every=25):
    # resume from a previous run if a progress file exists
    try:
        scraped_rows = pd.read_csv(progress_path).to_dict("records")
    except FileNotFoundError:
        scraped_rows = []
    already_done = {row["Artist Name"] for row in scraped_rows}

    coordinate_cache = {}
    total_artists = len(artists_dataset)
    for position, (artist_name, artist_type) in enumerate(
        zip(artists_dataset["Artist Name"], artists_dataset["Artist Type"]), start=1
    ):
        if artist_name in already_done:
            continue
        scraped = scrape_artist_location(artist_name, artist_type, coordinate_cache)
        scraped_rows.append({"Artist Name": artist_name, **scraped})
        print(f"[{position}/{total_artists}] {artist_name} -> "
              f"{scraped['Birthplace']} ({scraped['Scrape Status']})")
        if position % save_every == 0:                       # saves progress regularly
            pd.DataFrame(scraped_rows).to_csv(progress_path, index=False)

    scraped_locations = pd.DataFrame(scraped_rows)
    scraped_locations.to_csv(progress_path, index=False)
    return artists_dataset.merge(scraped_locations, on="Artist Name", how="left")

- **"scrape_artist_location"**: runs the full process for one artist/band (find the page, extract the birthplace, get the coordinates) and returns a result with the status reached: "no_article" for no music page found, "no_birthplace" for page found without finding place, "no_coordinates" for place found without finding coordinates or "ok". Recording how far each artist got makes it possible to analyse the failures later.

- **"add_locations_to_dataset"**: uses the cleaned dataset from section 2.1 ("artist_streams_cleaned.csv"), where each artist appears only once, so every artist is scraped a single time. The results are added to the dataset as new columns (a left merge, so no rows are lost). Progress is saved every 25 artists in "scrape_progress.csv", so if the run stops it resumes where it ended instead of starting over.

**Test of the scraper on the first 5 artists**

In [9]:
artists_clean = pd.read_csv("artist_streams_cleaned.csv")

print("CHECK BEFORE SCRAPING")
print("Rows:", len(artists_clean))
print("Duplicated names:", artists_clean["Artist Name"].duplicated().sum())
print("Missing names or types:", artists_clean[["Artist Name", "Artist Type"]].isna().sum().sum())

test_artists = artists_clean.head(5)
test_result = add_locations_to_dataset(test_artists, progress_path="scrape_test.csv")

print("TEST RESULT")
print(test_result[["Artist Name", "Birthplace", "Latitude", "Longitude",
                   "Match Quality", "Scrape Status"]].to_string())

CHECK BEFORE SCRAPING
Rows: 500
Duplicated names: 0
Missing names or types: 0
TEST RESULT
     Artist Name                             Birthplace  Latitude  Longitude Match Quality Scrape Status
0          Drake               Toronto, Ontario, Canada  43.65250  -79.38167        strong            ok
1   Taylor Swift         West Reading, Pennsylvania, US  40.33417  -75.94667        strong            ok
2      Bad Bunny                   Bayamón, Puerto Rico  18.38000  -66.16333        strong            ok
3     The Weeknd  Metropolitan Toronto, Ontario, Canada  43.70720  -79.38050        strong            ok
4  Justin Bieber                London, Ontario, Canada  42.98500  -81.24528        strong            ok


Before scraping the whole dataset, the functions were tested on 5 artists and the results were compared with their Wikipedia pages by hand.

| Artist | Birthplace in Wikipedia | Birthplace scraped | Coordinates match? |
|---|---|---|---|
| Drake | Toronto, Ontario, Canada | Toronto, Ontario, Canada | Yes |
| Taylor Swift | West Reading, Pennsylvania, US | West Reading, Pennsylvania, US | Yes |
| Bad Bunny | Bayamón, Puerto Rico | Bayamón, Puerto Rico | Yes |
| The Weeknd | Metropolitan Toronto, Ontario, Canada | Metropolitan Toronto, Ontario, Canada | Yes, but see below |
| Justin Bieber | London, Ontario, Canada | London, Ontario, Canada | Yes |

All 5 artists were matched to the correct page ("Match Quality" = strong) and obtained coordinates ("Scrape Status" = ok). The results also agree with the column "Country of Origin" of the original dataset: Canada for Drake, The Weeknd and Justin Bieber and United States for Taylor Swift and Bad Bunny, since Puerto Rico is considered as a US territory.

One difference was found: The Weeknd's birthplace is written as "Metropolitan Toronto", which is linked to a different Wikipedia page from "Toronto". As a result, his coordinates (43.7072, -79.3805) differ from Drake's (43.6525, -79.3817), even though both were born in the same city. This is the exact information that Wikipedia provides, so it was kept, and the difference is too small to be visible on a world map. It shows, however, that the coordinates depend on the page linked in each infobox.

**Scraping of the whole dataset and summary of the results**

In [10]:
artists_with_locations = add_locations_to_dataset(artists_clean)
artists_with_locations["Birthplace"] = artists_with_locations["Birthplace"].str.replace(r"\s+\)", ")", regex=True)
artists_with_locations.to_csv("artist_streams_with_locations.csv", index=False)

print("SCRAPING SUMMARY")
print(artists_with_locations["Scrape Status"].value_counts())
print()
print("MATCH QUALITY")
print(artists_with_locations["Match Quality"].value_counts(dropna=False))

SCRAPING SUMMARY
Scrape Status
ok                454
no_article         23
no_coordinates     22
no_birthplace       1
Name: count, dtype: int64

MATCH QUALITY
Match Quality
strong    472
NaN        23
weak        5
Name: count, dtype: int64


The scraper was run on all 500 artists of the cleaned dataset on 10 November 2026. Since Wikipedia pages change over time, running it on another date may give slightly different results. The results were saved in "artist_streams_with_locations.csv", which is used to build the interactive maps. Before saving, the stray space that some birthplaces had before a closing bracket was removed (e.g. "(now Kosovo )" became "(now Kosovo)").

| Step reached | Artists | % of 500 |
|---|---|---|
| Wikipedia page found | 477 | 95.4% |
| Birthplace or origin found | 476 | 95.2% |
| Coordinates found ("ok") | 454 | 90.8% |

Of the 477 pages found, 472 are "strong" matches (94.4% of all artists), where a name of the page is exactly the artist's name, and 5 are "weak" matches (1.0%), where one name contains the other. The 23 artists without a page have no match quality.

These are the final results, obtained after the matching rules were improved (see the review of the weak matches below). The first run reached 456 artists with coordinates, slightly more than now, but several of those locations came from pages of the wrong artist. The final number is therefore a little lower but reliable.

**Artists that did not get coordinates, grouped by the step where they failed**

In [11]:
failed_artists = artists_with_locations[artists_with_locations["Scrape Status"] != "ok"]
print("Artists without coordinates:", len(failed_artists))
for scrape_status, status_group in failed_artists.groupby("Scrape Status"):
    print(f"\n{scrape_status} ({len(status_group)}):")
    print(", ".join(status_group["Artist Name"]))

Artists without coordinates: 46

no_article (23):
Die drei ???, Disney, Henrique & Juliano, 2Pac, Childish Gambino, Amitabh Bhattacharya, MC Ryan SP, Irshad Kamil, Zé Neto & Cristiano, Paulo Londra, La Arrolladora Banda El Limón De Rene Camacho, League of Legends, Alka Yagnik, Mambo Kingz, Bibi Blocksberg, Eslabon Armado, Kumaar, Maiara & Maraisa, Mc IG, Grupo Menos É Mais, Magenta, Grupo Marca Registrada, benny blanco

no_birthplace (1):
Bee Gees

no_coordinates (22):
Harry Styles, Junior H, Lil Peep, Swae Lee, Duki, Anirudh Ravichander, Bryson Tiller, Becky G, Jack Harlow, Ava Max, Frank Sinatra, Robin Schulz, Chris Stapleton, Zedd, Kehlani, Maria Becerra, Florida Georgia Line, Julión Álvarez y su Norteño Banda, Gabito Ballesteros, Bonez MC, ScHoolboy Q, H.E.R.


46 artists ended without coordinates, at three different steps:

- **"no_article" (23):** no page was accepted for these artists. Four of them are not music artists (see the last cell of this section). The others probably have no page in the English Wikipedia or have a page that the rules could not match. For example, lyricists such as Amitabh Bhattacharya and Irshad Kamil, whose pages may not use the infobox of a musician, Brazilian duos such as Henrique & Juliano and Zé Neto & Cristiano and artists whose article uses their real name instead of their stage name, such as 2Pac (Tupac Shakur) and Childish Gambino (Donald Glover). With the stricter rules, it was preferred to leave these artists without a location rather than accept the page of another artist, something that happened in the first run.

- **"no_birthplace" (1):** the page of the Bee Gees was found, but no place could be read from the "Born" or "Origin" rows of its infobox.

- **"no_coordinates" (22):** the birthplace was found (e.g. for Harry Styles or Lil Peep), but no coordinates could be extracted from the page of the place, from the broader regions linked in the infobox or from the search fallback, either because those pages have no coordinates or because they are written in a format that the code does not read.

**Music pages but the title differs from the artist's name**

In [12]:
weak_matches = artists_with_locations[artists_with_locations["Match Quality"] == "weak"]
print("WEAK MATCHES TO REVIEW:", len(weak_matches))
print(weak_matches[["Artist Name", "Wikipedia Page", "Birthplace", "Country of Origin"]].to_string())

WEAK MATCHES TO REVIEW: 5
                           Artist Name                                          Wikipedia Page                         Birthplace Country of Origin
63                              Pritam        https://en.wikipedia.org/wiki/Pritam_Chakraborty       Calcutta, West Bengal, India             India
208       Banda MS de Sergio Lizárraga                  https://en.wikipedia.org/wiki/Banda_MS          Mazatlán, Sinaloa, México            Mexico
235                               ZAYN                https://en.wikipedia.org/wiki/Zayn_Malik  Bradford, West Yorkshire, England    United Kingdom
340  Julión Álvarez y su Norteño Banda  https://en.wikipedia.org/wiki/Juli%C3%B3n_%C3%81lvarez      La Concordia, Chiapas, Mexico            Mexico
449                               Zion           https://en.wikipedia.org/wiki/Zion_%26_Lennox              Carolina, Puerto Rico     United States


In the first run, when no page had exactly the artist's name, the first music page found was accepted as a "weak" match. This gave 30 weak matches, which were all opened and checked by hand. Only 10 were the correct artist under a different title (e.g. "ZAYN" and "Zayn Malik"). The other 20 were wrong: 4 album pages (e.g. Ty Dolla $ign was matched to the album "Vultures 2"), 12 pages of other artists (e.g. 2Pac was matched to Danny Boy) and 4 entries that are not musicians and were matched to real musicians (e.g. Disney to Pogo).

Based on this review, the matching rules were improved (explained with the functions above): symbols are written as letters, the names in bold in the first paragraph of the article are also compared, a weak match is only accepted when one name contains the other, and album and song pages are rejected. The weak matches were then scraped again.

With the new rules, 5 of the previously correct artists became "strong" matches (A$AP Rocky, \$uicideboy\$, P!nk, Bob Marley & The Wailers and Florence + The Machine), and all 20 wrong pages were rejected. One of these artists, Ty Dolla \$ign, was then found on his correct page, because the "$" in his name is now read as "s". The other 19 ended without a page ("no_article"), which is why this status went from 4 artists in the first run to 23. The 5 weak matches that remain were checked by hand again, and all of them are the correct artist:

| Artist in the dataset | Wikipedia page | Why the names differ |
|---|---|---|
| Pritam | Pritam Chakraborty | Page uses his full name |
| Banda MS de Sergio Lizárraga | Banda MS | Page uses the short name of the band |
| ZAYN | Zayn Malik | Page uses his full name |
| Julión Álvarez y su Norteño Banda | Julión Álvarez | Page of the band's leader, after whom the band is named |
| Zion | Zion & Lennox | Page of the duo he belongs to (same birthplace, Carolina, Puerto Rico) |

For this reason, no location had to be removed by hand.

**Comparing the last part of the scraped birthplace with the dataset's country**

In [13]:
# names that mean the same country
country_aliases = {
    "us": "united states", "u.s.": "united states", "u.s": "united states", "usa": "united states",
    "united states of america": "united states", "puerto rico": "united states",
    "uk": "united kingdom", "england": "united kingdom", "scotland": "united kingdom",
    "wales": "united kingdom", "northern ireland": "united kingdom",
    "republic of ireland": "ireland", "democratic republic of the congo": "dr congo",
    "korea": "south korea", "republic of korea": "south korea",
    "méxico": "mexico", "west germany": "germany",                     
    "soviet union": "russia", "russian sfsr": "russia",                  
}

def standard_country(country_text):
    country_key = str(country_text).strip().lower()
    return country_aliases.get(country_key, country_key)

located_artists = artists_with_locations[artists_with_locations["Birthplace"].notna()].copy()
located_artists["Scraped Country"] = located_artists["Birthplace"].str.split(",").str[-1].str.strip()
located_artists["Country Agrees"] = (
    located_artists["Scraped Country"].map(standard_country)
    == located_artists["Country of Origin"].map(standard_country))
country_mismatches = located_artists[~located_artists["Country Agrees"]]

print("COUNTRY CHECK")
print("Artists with a birthplace:", len(located_artists))
print("Country agrees with the dataset:", located_artists["Country Agrees"].sum())
print("To review by hand:", len(country_mismatches))
print(country_mismatches[["Artist Name", "Birthplace", "Country of Origin", "Match Quality"]].to_string())

COUNTRY CHECK
Artists with a birthplace: 476
Country agrees with the dataset: 461
To review by hand: 15
         Artist Name                             Birthplace Country of Origin Match Quality
47           J. Cole                Frankfurt, West Germany     United States        strong
49               Sia              Adelaide, South Australia         Australia        strong
105       Zach Bryan                        Yokosuka, Japan     United States        strong
106             Kygo                              Singapore            Norway        strong
125       Kali Uchis             Alexandria, Virginia, U.S.          Colombia        strong
132      Luis Miguel                  San Juan, Puerto Rico            Mexico        strong
134             Akon              St. Louis, Missouri, U.S.           Senegal        strong
138      Alan Walker                   Northampton, England            Norway        strong
155  Black Eyed Peas                Los Angeles, California     Unit

As a second check, the country at the end of each scraped birthplace was compared with the column "Country of Origin" of the dataset. A difference does not always mean an error, so all differences were reviewed by hand.

In the first run, this check found 29 differences, 9 of which were wrong pages. Two of these had a "strong" match, because another artist has exactly the same name: "Dave" was matched to a Dutch singer called Dave, born in Amsterdam, instead of the British rapper, and "SEVENTEEN" was matched to an Indonesian band instead of the South Korean group. For these two artists, the exact Wikipedia title ("Dave (rapper)" and "Seventeen (South Korean band)") was given to the search box. Only the search term was given: the birthplace and the coordinates were still obtained by scraping. Other spellings and historical names of countries (e.g. "México", "West Germany" and "Soviet Union") were also added to the comparison, so they are no longer reported as differences.

In the final results, 461 of the 476 artists with a birthplace (96.8%) agree with the dataset. The 15 remaining differences are all correct pages:
- 11 artists were born in a different country from the one in the dataset (e.g. J. Cole, Zach Bryan and ScHoolboy Q were born on US military bases abroad, and Akon was born in the United States although the dataset lists Senegal);
- in 4 cases, the birthplace is written without the country (e.g. "Los Angeles, California").

For Udit Narayan, Wikipedia gives a birthplace in India while the dataset lists Nepal. The scraped value was kept because it is the information provided by Wikipedia.

**Entries of the dataset that are not music artists**

In [14]:
# no birthplace exists for them
non_artist_entries = ["Disney", "League of Legends", "Die drei ???", "Bibi Blocksberg"]

print("NON-ARTIST ENTRIES")
print(artists_with_locations.loc[artists_with_locations["Artist Name"].isin(non_artist_entries),
      ["Artist Name", "Primary Genre", "Wikipedia Page", "Birthplace", "Scrape Status"]].to_string())

NON-ARTIST ENTRIES
           Artist Name     Primary Genre Wikipedia Page Birthplace Scrape Status
81        Die drei ???       Spoken Word            NaN        NaN    no_article
89              Disney        Soundtrack            NaN        NaN    no_article
320  League of Legends        Soundtrack            NaN        NaN    no_article
403    Bibi Blocksberg  Children's Music            NaN        NaN    no_article


Four entries of the dataset are not music artists: "Disney" (a company, with soundtrack releases), "League of Legends" (a video game, with its soundtrack), "Die drei ???" (a German audio-drama series) and "Bibi Blocksberg" (a cartoon and audio-drama character). This is also visible in their genres, which are "Soundtrack", "Spoken Word" and "Children's Music".

In the first run, all four were matched to pages of real musicians (e.g. Die drei ??? to the band Trio and Bibi Blocksberg to Tom Keenlyside), which gave them wrong locations. With the improved rules, no page was accepted for them, so they end with the status "no_article" and without a location. This is correct, because no birthplace exists for them.

**Conclusion of the web scraping**

Starting from the Wikipedia Main Page, the birthplace or origin and the coordinates were obtained for 454 of the 500 artists (90.8%). The results were verified in two ways: every weak match was opened and checked by hand, and every scraped country was compared with the column "Country of Origin" of the dataset. This review showed that the first rules accepted the pages of wrong artists, so the rules were improved and the affected artists were scraped again. As a result, all locations in the final dataset come from the correct page.

The main limitations are that 46 artists have no coordinates (mostly artists without a matching English page, or places whose coordinates could not be read) and that the coordinates depend on the page linked in each infobox, as seen with The Weeknd ("Metropolitan Toronto"). The file "artist_streams_with_locations.csv" is the input for the interactive maps in the next section.